# Network Markowitz Implementation (Giudici et al., 2020)
## Implementasi Paper: Network Models to Improve Automated Cryptocurrency Portfolio Management

Paper ini mengusulkan kombinasi:
1. **Random Matrix Theory (RMT)** - filter noise dari correlation matrix
2. **Minimal Spanning Tree (MST)** - simplifikasi hubungan antar aset
3. **Network Centrality** - ukuran kepentingan aset dalam jaringan
4. **Markowitz Optimization** - dengan penalty berdasarkan centrality

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.linalg import eigh
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Generate Dummy Cryptocurrency Data

In [ ]:
# Simulasi data 10 cryptocurrency seperti dalam paper
np.random.seed(42)

n_assets = 10
n_days = 764  # Seperti dalam paper (14 Sept 2017 - 17 Okt 2019)

crypto_names = ['BTC', 'ETH', 'XRP', 'USDT', 'BCH', 'LTC', 'BNB', 'EOS', 'XLM', 'TRX']

# Generate correlated returns
mean_returns = np.random.uniform(-0.001, 0.003, n_assets)
volatilities = np.array([0.04, 0.05, 0.07, 0.01, 0.08, 0.06, 0.07, 0.07, 0.10, 0.15])  # Dari paper

# Create correlation structure (crypto markets are highly correlated)
base_corr = 0.6
corr_matrix = np.full((n_assets, n_assets), base_corr)
np.fill_diagonal(corr_matrix, 1.0)
corr_matrix[3, :] = 0.1  # USDT (stablecoin) low correlation
corr_matrix[:, 3] = 0.1
corr_matrix[3, 3] = 1.0

# Generate covariance matrix
cov_matrix = np.outer(volatilities, volatilities) * corr_matrix

# Generate returns
returns = np.random.multivariate_normal(mean_returns, cov_matrix, n_days)
df_returns = pd.DataFrame(returns, columns=crypto_names)

print("Data Summary:")
print(df_returns.describe())
print(f"\nShape: {df_returns.shape}")

## 2. Random Matrix Theory (RMT) - Filter Correlation Matrix

In [ ]:
def apply_rmt_filter(returns_data):
    """
    Apply Random Matrix Theory filtering (Marchenko-Pastur)
    """
    T, N = returns_data.shape
    Q = T / N
    
    # Compute correlation matrix
    C = np.corrcoef(returns_data.T)
    
    # Eigenvalue decomposition
    eigenvalues, eigenvectors = eigh(C)
    eigenvalues = eigenvalues[::-1]  # Sort descending
    eigenvectors = eigenvectors[:, ::-1]
    
    # Marchenko-Pastur bounds
    lambda_plus = 1 + (1/Q) + 2*np.sqrt(1/Q)
    lambda_minus = 1 + (1/Q) - 2*np.sqrt(1/Q)
    
    print(f"Q = T/N = {Q:.2f}")
    print(f"λ+ (threshold) = {lambda_plus:.4f}")
    print(f"λ- = {lambda_minus:.4f}")
    print(f"\nEigenvalues: {eigenvalues}")
    
    # Filter: keep only eigenvalues > lambda_plus
    significant_mask = eigenvalues > lambda_plus
    n_significant = significant_mask.sum()
    
    print(f"\nSignificant eigenvalues: {n_significant}/{N}")
    
    # Reconstruct filtered correlation matrix
    Lambda_filtered = np.diag(np.where(significant_mask, eigenvalues, 0))
    C_filtered = eigenvectors @ Lambda_filtered @ eigenvectors.T
    
    return C_filtered, eigenvalues, lambda_plus

# Apply RMT on a window (120 days as in paper)
window_data = df_returns.iloc[:120]
C_filtered, eigenvals, threshold = apply_rmt_filter(window_data)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original correlation
sns.heatmap(np.corrcoef(window_data.T), annot=True, fmt='.2f', cmap='coolwarm', 
            xticklabels=crypto_names, yticklabels=crypto_names, ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Original Correlation Matrix')

# Filtered correlation
sns.heatmap(C_filtered, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=crypto_names, yticklabels=crypto_names, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('RMT Filtered Correlation Matrix')

# Eigenvalue spectrum
axes[2].bar(range(len(eigenvals)), eigenvals, alpha=0.7)
axes[2].axhline(y=threshold, color='r', linestyle='--', label=f'λ+ = {threshold:.2f}')
axes[2].set_xlabel('Eigenvalue Index')
axes[2].set_ylabel('Eigenvalue')
axes[2].set_title('Eigenvalue Spectrum')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Minimal Spanning Tree (MST)

In [ ]:
def build_mst(correlation_matrix, asset_names):
    """
    Build Minimal Spanning Tree from correlation matrix
    Distance: d_ij = sqrt(2 - 2*c_ij)
    """
    # Convert correlation to distance
    distance_matrix = np.sqrt(2 - 2*correlation_matrix)
    np.fill_diagonal(distance_matrix, 0)
    
    # Compute MST
    mst = minimum_spanning_tree(distance_matrix)
    mst_full = mst.toarray()
    
    # Create NetworkX graph for visualization
    G = nx.Graph()
    for i, name in enumerate(asset_names):
        G.add_node(name)
    
    for i in range(len(asset_names)):
        for j in range(i+1, len(asset_names)):
            if mst_full[i, j] > 0 or mst_full[j, i] > 0:
                weight = max(mst_full[i, j], mst_full[j, i])
                G.add_edge(asset_names[i], asset_names[j], weight=weight)
    
    return G, distance_matrix

# Build MST from filtered correlation
G_mst, dist_matrix = build_mst(C_filtered, crypto_names)

# Visualize MST
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_mst, k=2, iterations=50, seed=42)
nx.draw_networkx_nodes(G_mst, pos, node_size=3000, node_color='lightblue', 
                       edgecolors='black', linewidths=2)
nx.draw_networkx_labels(G_mst, pos, font_size=12, font_weight='bold')
nx.draw_networkx_edges(G_mst, pos, width=2, alpha=0.6)

edge_labels = nx.get_edge_attributes(G_mst, 'weight')
edge_labels = {k: f"{v:.2f}" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G_mst, pos, edge_labels, font_size=9)

plt.title('Minimal Spanning Tree (MST) of Cryptocurrencies', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nMST has {G_mst.number_of_edges()} edges (should be {len(crypto_names)-1})")

## 4. Network Centrality (Eigenvector Centrality)

In [ ]:
def compute_eigenvector_centrality(distance_matrix):
    """
    Compute eigenvector centrality from distance matrix
    Higher centrality = more different from others (higher systemic risk)
    """
    # Create adjacency matrix (inverse of distance for centrality)
    adjacency = 1 / (distance_matrix + 1e-8)
    np.fill_diagonal(adjacency, 0)
    
    # Compute eigenvector centrality
    eigenvalues, eigenvectors = eigh(adjacency)
    principal_eigenvector = np.abs(eigenvectors[:, -1])
    
    # Normalize
    centrality = principal_eigenvector / principal_eigenvector.sum()
    
    return centrality

centrality_scores = compute_eigenvector_centrality(dist_matrix)

# Visualize centrality
centrality_df = pd.DataFrame({
    'Asset': crypto_names,
    'Centrality': centrality_scores
}).sort_values('Centrality', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(centrality_df['Asset'], centrality_df['Centrality'], color='steelblue', edgecolor='black')
plt.xlabel('Eigenvector Centrality', fontsize=12)
plt.ylabel('Cryptocurrency', fontsize=12)
plt.title('Network Centrality Scores (Higher = More Systemic Risk)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCentrality Scores:")
print(centrality_df.to_string(index=False))

## 5. Portfolio Optimization - Network Markowitz

### Objective Function:
$$\min_w w^T \Sigma^* w + \gamma \sum_{i=1}^{n} x_i w_i$$

Where:
- $\Sigma^*$ = filtered covariance matrix (RMT + MST)
- $\gamma$ = systemic risk aversion parameter
- $x_i$ = eigenvector centrality of asset i

In [ ]:
def network_markowitz_optimization(returns_data, correlation_filtered, centrality, gamma=0):
    """
    Network Markowitz Portfolio Optimization
    """
    n_assets = returns_data.shape[1]
    
    # Compute filtered covariance matrix
    std_devs = returns_data.std().values
    cov_filtered = np.outer(std_devs, std_devs) * correlation_filtered
    
    # Mean returns
    mean_returns = returns_data.mean().values
    
    # Objective function: portfolio variance + centrality penalty
    def objective(w):
        portfolio_variance = w @ cov_filtered @ w
        centrality_penalty = gamma * np.sum(centrality * w)
        return portfolio_variance + centrality_penalty
    
    # Constraints
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # weights sum to 1
        {'type': 'ineq', 'fun': lambda w: w @ mean_returns - mean_returns.mean()}  # min return constraint
    ]
    
    # Bounds: w_i >= 0 (no short selling)
    bounds = tuple((0, 1) for _ in range(n_assets))
    
    # Initial guess: equal weights
    w0 = np.ones(n_assets) / n_assets
    
    # Optimize
    result = minimize(objective, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    
    if result.success:
        weights = result.x
        portfolio_return = weights @ mean_returns
        portfolio_risk = np.sqrt(weights @ cov_filtered @ weights)
        return weights, portfolio_return, portfolio_risk
    else:
        print(f"Optimization failed: {result.message}")
        return None, None, None

# Test different gamma values (as in paper)
gamma_values = [0, 0.005, 0.025, 0.05, 0.15, 0.7, 1.0]
results = []

for gamma in gamma_values:
    weights, ret, risk = network_markowitz_optimization(window_data, C_filtered, centrality_scores, gamma)
    if weights is not None:
        results.append({
            'gamma': gamma,
            'weights': weights,
            'return': ret * 252,  # Annualized
            'risk': risk * np.sqrt(252),  # Annualized
            'sharpe': (ret * 252) / (risk * np.sqrt(252)) if risk > 0 else 0
        })

# Display results
print("\n" + "="*80)
print("NETWORK MARKOWITZ OPTIMIZATION RESULTS")
print("="*80)

for res in results:
    print(f"\nγ = {res['gamma']:.3f}")
    print(f"  Annual Return: {res['return']*100:.2f}%")
    print(f"  Annual Risk:   {res['risk']*100:.2f}%")
    print(f"  Sharpe Ratio:  {res['sharpe']:.4f}")
    print(f"  Top 3 Holdings:")
    top_idx = np.argsort(res['weights'])[-3:][::-1]
    for idx in top_idx:
        print(f"    {crypto_names[idx]}: {res['weights'][idx]*100:.2f}%")

## 6. Visualization: Portfolio Weights Across Different γ

In [ ]:
# Create weights dataframe
weights_df = pd.DataFrame(
    [res['weights'] for res in results],
    columns=crypto_names,
    index=[f"γ={res['gamma']}" for res in results]
)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 7))
weights_df.T.plot(kind='bar', stacked=False, ax=ax, width=0.8, colormap='tab10')
ax.set_xlabel('Cryptocurrency', fontsize=12, fontweight='bold')
ax.set_ylabel('Portfolio Weight', fontsize=12, fontweight='bold')
ax.set_title('Portfolio Weights Across Different Risk Aversion Parameters (γ)', 
             fontsize=14, fontweight='bold')
ax.legend(title='Risk Aversion', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nPortfolio Weights Table:")
print(weights_df.round(4))

## 7. Efficient Frontier Comparison

In [ ]:
# Plot efficient frontier
plt.figure(figsize=(10, 7))

risks = [res['risk'] for res in results]
returns = [res['return'] for res in results]
gammas = [res['gamma'] for res in results]

scatter = plt.scatter(risks, returns, c=gammas, s=200, cmap='viridis', 
                     edgecolors='black', linewidths=2, alpha=0.8)

# Annotate points
for i, (r, ret, g) in enumerate(zip(risks, returns, gammas)):
    plt.annotate(f'γ={g}', (r, ret), xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.colorbar(scatter, label='Risk Aversion (γ)')
plt.xlabel('Annual Risk (Volatility)', fontsize=12, fontweight='bold')
plt.ylabel('Annual Return', fontsize=12, fontweight='bold')
plt.title('Network Markowitz: Risk-Return Profile', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary & Key Insights

### Metodologi Paper:
1. **RMT Filter**: Menghilangkan noise dari correlation matrix
2. **MST**: Menyederhanakan struktur jaringan menjadi N-1 edges
3. **Centrality**: Mengukur systemic risk setiap aset
4. **Optimization**: Markowitz + centrality penalty

### Parameter γ (Risk Aversion):
- **γ = 0**: Network Markowitz (hanya RMT+MST, no centrality penalty)
  - Terbaik untuk **bear market** (proteksi downside)
- **γ > 0**: Dengan centrality penalty
  - Terbaik untuk **bull market** (adaptif terhadap kondisi)
  - Semakin besar γ → lebih avoid aset dengan high centrality (high systemic risk)

### Temuan Paper:
- Bull market: Model dengan γ > 0 beradaptasi cepat
- Bear market: γ = 0 memberikan proteksi terbaik
- VaR lebih rendah dibanding benchmark
- Sharpe ratio kompetitif